# 01 · Explore the incubator history

One device, one day (3 Oct 2026), 1 367 readings at 15 s. The same functions are used by `scripts/make_figures.py`, so anything you see here is what the README reports.

In [ ]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
import matplotlib.pyplot as plt
from incubator.load import load_history, load_stress_log, summary
from incubator import features as fe, anomaly as an, thermal_fit as tf, faults as fl
pd.set_option("display.width", 160)

In [ ]:
df = load_history()
summary(df)

In [ ]:
df.head()

## Timeline
Shaded spans would be heater-on; here we just plot the three temperatures.

In [ ]:
ax = df[['temp', 'temp_out', 'target']].plot(figsize=(11, 4), lw=1.2)
ax.set_ylabel('°C');

## Heater runs
Each row is a maximal run with constant heater state.

In [ ]:
seg = fe.heater_segments(df)
seg[seg['duration_min'] >= 2]

In [ ]:
fe.heating_rates(seg).groupby('heater')['rate_c_per_min'].describe()

## Duty cycle, setpoint steps, time-to-target and overshoot

In [ ]:
dc = fe.duty_cycle(df, '10min')
dc.plot.bar(figsize=(11, 3), width=0.9)
plt.xticks([]);
dc.describe()

In [ ]:
fe.time_to_target(df)

## Thermal lag after switch-off
`post_switch_rise` is the extra warming in the 2 min after the heater goes off; look at the long runs.

In [ ]:
seg[seg['heater']].plot.scatter(x='duration_min', y='post_switch_rise', figsize=(6, 4));

## Humidity
Relative humidity falls when air is warmed at constant moisture, so look at absolute humidity (g/m³) to see whether the chamber dries out. It does not: inside, vapour density *rises* with temperature while RH stays in a narrow band.

In [ ]:
h = fe.humidity_table(df)
h[['rh_in', 'rh_out', 'ah_in', 'ah_out']].plot(figsize=(11, 4), lw=1.2, subplots=[('rh_in', 'rh_out'), ('ah_in', 'ah_out')], layout=(1, 2));

In [ ]:
h.resample('30min').mean().round(1)

## The heater test log (10 Oct)
Only ~2 h of 2-minute polls, but its heat-up slope can be compared with the 3 Oct history.

In [ ]:
stress = load_stress_log()
stress[['phase', 'temp', 'target', 'heater', 'run_state', 'action']]